# High-Performance VQA Fine-tuning (Target: 0.9+)

이 노트북은 **Qwen2.5-VL-3B-Instruct** 모델을 사용하여 시각적 질의응답(VQA) 성능을 극대화하기 위해 제작되었습니다.

### 🚀 주요 성능 향상 포인트:
1. **Label Masking (핵심)**: 질문과 이미지 토큰은 학습에서 제외하고, 모델이 내뱉는 **'정답(a, b, c, d)' 부분에 대해서만 Loss를 계산**합니다. 이 방식은 모델이 불필요한 언어 패턴을 학습하는 것을 막고 정답 정확도를 비약적으로 높입니다.
2. **Full Dataset**: 200개 샘플이 아닌 전체 데이터를 사용하여 학습합니다.
3. **LoRA Rank UP (32)**: 모델의 시각적 이해 능력을 높이기 위해 LoRA 가중치를 늘렸습니다.
4. **Image Resolution 상향**: 이미지 크기를 448로 높여 세밀한 객체 인식을 돕습니다.

In [ ]:
# 1. 필수 라이브러리 설치
!pip -q install "transformers>=4.43.2" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade

In [ ]:
import os, math, random
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

# 하이퍼파라미터 설정
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 448 
EPOCHS = 3
LR = 1e-4
GRAD_ACCUM = 8 
SEED = 42

random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 데이터 로드 (경로 확인 필수)
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
print(f"Total training data: {len(train_df)}")

In [ ]:
# 2. 모델 로드 및 LoRA 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=32,           
    lora_alpha=64,  
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 3. 데이터셋 및 Label Masking Collator
class VQAHighPerfDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        prompt = f"{row['question']}\n(a) {row['a']}\n(b) {row['b']}\n(c) {row['c']}\n(d) {row['d']}\n정답:"
        answer = str(row["answer"]).strip().lower()

        messages = [
            {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]},
            {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        ]
        return {"messages": messages, "image": img}

@dataclass
class HighPerfCollator:
    processor: Any
    def __call__(self, batch):
        texts = [self.processor.apply_chat_template(s["messages"], tokenize=False) for s in batch]
        images = [s["image"] for s in batch]
        
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        
        # Label Masking: 정답 부분(마지막 토큰들)만 학습하고 나머지는 -100 처리
        labels = enc["input_ids"].clone()
        for i in range(labels.shape[0]):
            # Qwen2-VL의 assistant 응답은 마지막 부분에 위치함
            # 정답 한 글자이므로 마지막 2~3개 토큰을 제외하고 마스킹
            labels[i, :-2] = -100 
            
        enc["labels"] = labels
        return enc

split = int(len(train_df) * 0.98)
train_ds = VQAHighPerfDataset(train_df.iloc[:split], processor)
valid_ds = VQAHighPerfDataset(train_df.iloc[split:], processor)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=HighPerfCollator(processor))
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=HighPerfCollator(processor))

In [ ]:
# 4. 학습 실행
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_training_steps = EPOCHS * (len(train_loader) // GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(optimizer, int(num_training_steps*0.1), num_training_steps)
scaler = torch.cuda.amp.GradScaler()

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    running_loss = 0.0
    
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM
        
        scaler.scale(loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            pbar.set_postfix({"loss": f"{running_loss:.4f}"})
            running_loss = 0.0

model.save_pretrained("qwen_best_lora")
print("Training Complete!")

In [ ]:
# 5. 추론 및 제출 파일 생성
model.eval()
results = []

for i in tqdm(range(len(test_df)), desc="Inference"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    prompt = f"{row['question']}\n(a) {row['a']}\n(b) {row['b']}\n(c) {row['c']}\n(d) {row['d']}\n정답:"
    
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=2)
    
    output = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # 마지막 한 글자 추출
    ans = output.strip().lower()[-1] 
    if ans not in ['a', 'b', 'c', 'd']: ans = 'a' # 예외 처리
    results.append(ans)

submission = pd.DataFrame({"id": test_df["id"], "answer": results})
submission.to_csv("submission_high_perf.csv", index=False)
print("Submission file saved!")